In [20]:
import numpy as np
import pandas as pd
from datetime import date
pd.set_option("display.max_rows", 100)

In [61]:
n = 500
r = 4
all_diagnoses = pd.DataFrame(
    {
        "PATIENT_ID": np.random.randint(0, n // r, n),
        "VISIT_DATE": pd.to_datetime(
            np.random.randint(
                pd.Timestamp("2015-01-01").value // 10**9,
                pd.Timestamp("2021-01-01").value // 10**9,
                n
            ),
            unit="s"
        ).strftime("%Y-%m-%d"),
        "ICD10_CODE": np.random.choice(["J01", "K02", "K03", "L01"], n),
        "DOCTOR_ID": np.random.randint(0, n // r, n),
    }
).sort_values("VISIT_DATE")

diagnosis = all_diagnoses.groupby("PATIENT_ID").first().reset_index()

prescription = pd.DataFrame({
    "PATIENT_ID": np.random.randint(0, n // r, n),
    "PRESCRIPTION_DATE": pd.to_datetime(
        np.random.randint(
            pd.Timestamp("2013-01-01").value // 10**9,
            pd.Timestamp("2020-01-01").value // 10**9,
            n
        ),
        unit="s"
    ).strftime("%Y-%m-%d"),
    "DOCTOR_ID": np.random.randint(0, n // r + 10, n),
    "CODE": np.random.choice(["A01", "B01", "B02", "C01"], n),
})
prescription = prescription.sort_values("PRESCRIPTION_DATE")
prescription.to_csv("prescription.csv", index=False)
diagnosis.to_csv("diagnosis.csv", index=False)

In [62]:
date_column = "PRESCRIPTION_DATE"
events = prescription
group_column = "CODE_GROUP"
code_column = "CODE"
events_file = "prescription.csv"
DIAGNOSIS_DTYPES = {"PATIENT_ID": "str", "VISIT_DATE": "str", "ICD10_CODE": "str", "DOCTOR_ID": "str"}
events_dtypes = {
    "PATIENT_ID": "str",
    "DOCTOR_ID": "str",
    "CODE": "str",
    "PRESCRIPTION_DATE": "str",
}
CHUNK_SIZE = 10
history_column_prefix = "GOT"

In [63]:
diagnosis = pd.read_csv("diagnosis.csv", dtype=DIAGNOSIS_DTYPES)

earliest_events_dict = {}
code_groups_set = set()

for chunk in pd.read_csv(events_file, dtype=events_dtypes, usecols=events_dtypes.keys(), chunksize=CHUNK_SIZE):
    chunk = chunk[chunk[code_column].str.match("^[A-Z]", na=False)]  # Clean up improper codes
    chunk[group_column] = chunk[code_column].str[0]
    code_groups_set.update(chunk[group_column].unique())
    grouped = chunk.groupby(["PATIENT_ID", group_column])[date_column].min().reset_index()
    for _, row in grouped.iterrows():
        key = (row["PATIENT_ID"], row[group_column])
        date_val = row[date_column]
        if key not in earliest_events_dict or pd.to_datetime(date_val) < pd.to_datetime(earliest_events_dict[key]):
            earliest_events_dict[key] = date_val

code_groups = sorted(code_groups_set)
earliest_events = pd.DataFrame([{"PATIENT_ID": k[0], group_column: k[1], "EARLIEST_EVENT_DATE": v} for k, v in earliest_events_dict.items()])

doctor_ids = diagnosis["DOCTOR_ID"].unique()
full_index = pd.MultiIndex.from_product([doctor_ids, code_groups], names=["PATIENT_ID", group_column])
earliest_events = (
    earliest_events.set_index(["PATIENT_ID", group_column])
    .reindex(full_index, fill_value=pd.NA)
    .reset_index()
    .rename(columns={"PATIENT_ID": "DOCTOR_ID"})
    .sort_values(["DOCTOR_ID", group_column])
)
print("Earliest events generated.")

def add_history_columns(earliest_events, diagnosis, code_groups, group_column, history_column_prefix):
    doctor_event_history = (
        diagnosis[["DOCTOR_ID", "VISIT_DATE"]].drop_duplicates().sort_values(["DOCTOR_ID", "VISIT_DATE"]).reset_index(drop=True)
    )

    for group in code_groups:
        group_history = earliest_events[earliest_events[group_column] == group][["DOCTOR_ID", "EARLIEST_EVENT_DATE"]]
        merged = pd.merge(
            doctor_event_history,
            group_history,
            on="DOCTOR_ID",
            how="inner",
        )
        doctor_event_history[f"{history_column_prefix}_{group}"] = (merged["EARLIEST_EVENT_DATE"] < merged["VISIT_DATE"]).astype(int)
    return doctor_event_history

doctor_event_history = add_history_columns(earliest_events, diagnosis, code_groups, group_column, history_column_prefix)
doctor_event_history = doctor_event_history.sort_values(["DOCTOR_ID", "VISIT_DATE"])

Earliest events generated.


In [64]:
doctor_event_history.head(10)

,DOCTOR_ID,VISIT_DATE,GOT_A,GOT_B,GOT_C
0,0,2015-02-20,0,0,0
1,10,2015-01-27,0,1,1
2,100,2015-07-25,1,1,0
3,100,2015-09-06,1,1,0
4,101,2018-05-24,0,0,1
5,102,2019-01-15,0,1,1
6,103,2016-11-29,0,0,0
7,103,2018-09-06,1,1,0
8,104,2017-11-30,0,1,0
9,105,2015-07-19,0,0,1


In [65]:
prescription[prescription["PATIENT_ID"] == 103]

,PATIENT_ID,PRESCRIPTION_DATE,DOCTOR_ID,CODE
289,103,2017-07-30,4,B02
56,103,2017-10-25,119,A01


In [ ]:
def count_dates_before(doctor_event_history, doctor_dates):
    def count_before(doctor_id, visit_date):
        dates = doctor_dates.get(doctor_id)
        if dates is None or len(dates) == 0:
            return 0
        visit_date = np.datetime64(visit_date)
        return np.searchsorted(dates, visit_date, side="left")

    return [count_before(row["DOCTOR_ID"], row["VISIT_DATE"]) for _, row in doctor_event_history.iterrows()]


def add_mean_yearly_past_prescriptions_column(events, date_column, doctor_event_history, diagnosis):
    diagnosis["VISIT_DATE"] = pd.to_datetime(diagnosis["VISIT_DATE"])


    events[date_column] = pd.to_datetime(events[date_column])

    doctor_earliest_presc = events.groupby("DOCTOR_ID")[date_column].min()
    doctor_event_history["EARLIEST_PRESC_DATE"] = doctor_event_history["DOCTOR_ID"].map(doctor_earliest_presc)
    doctor_presc_dates = events.groupby("DOCTOR_ID")[date_column].apply(lambda x: pd.to_datetime(x).sort_values().to_numpy(dtype="datetime64[ns]"))
    doctor_diagnosis_dates = all_diagnoses.groupby("DOCTOR_ID")["VISIT_DATE"].apply(
        lambda x: pd.to_datetime(x).sort_values().to_numpy(dtype="datetime64[ns]")
    )
    doctor_event_history["N_PRESCRIPTIONS_BEFORE"] = count_dates_before(doctor_event_history, doctor_presc_dates)
    doctor_event_history["N_DIAGNOSES_BEFORE"] = count_dates_before(doctor_event_history, doctor_diagnosis_dates)
    doctor_event_history["N_DIAGNOSES_BEFORE"] = doctor_event_history["N_DIAGNOSES_BEFORE"].clip(lower=1)  # Prevent division by zero
    mask_valid = doctor_event_history["EARLIEST_PRESC_DATE"].notna() & (
        doctor_event_history["VISIT_DATE"] > doctor_event_history["EARLIEST_PRESC_DATE"]
    )
    years = (doctor_event_history["VISIT_DATE"] - doctor_event_history["EARLIEST_PRESC_DATE"]).dt.days / 365.25
    doctor_event_history["MEAN_YEARLY_PRESCRIPTIONS"] = pd.NA
    doctor_event_history.loc[mask_valid, "MEAN_YEARLY_PRESCRIPTIONS"] = (
        doctor_event_history.loc[mask_valid, "N_PRESCRIPTIONS_BEFORE"] / years[mask_valid]
    ).round(2)
    doctor_event_history["MEAN_PRESCRIPTIONS_PER_DIAGNOSIS"] = pd.NA
    doctor_event_history.loc[mask_valid, "MEAN_PRESCRIPTIONS_PER_DIAGNOSIS"] = (
        doctor_event_history.loc[mask_valid, "N_PRESCRIPTIONS_BEFORE"] / doctor_event_history.loc[mask_valid, "N_DIAGNOSES_BEFORE"]
    ).round(2)
    doctor_event_history.drop(columns=["N_PRESCRIPTIONS_BEFORE", "N_DIAGNOSES_BEFORE", "EARLIEST_PRESC_DATE"], inplace=True)